In [1]:
import pandas as pd 
import os
from timezonefinder import TimezoneFinder
from tqdm import tqdm

tqdm.pandas()

obj = TimezoneFinder()


In [21]:
df = pd.read_csv('metadata_v3.csv')
df.set_index("image_id")

,id,date,time,day_of_week,location,location_displayed,location_id,new_lat,new_lng,activity,activity_id,event_id,caption,ocr,object_tags,video_id,image_available
image_id,,,,,,,,,,,,,,,,,
201901/01/20190101_103717_000,1,2019-01-01,10:37,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.389980,-6.145760,watching tv,0,0,a window with curtains and a picture of a chair,'口',"curtain,house,bed",20190101,1
201901/01/20190101_103749_000,2,2019-01-01,10:37,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.389980,-6.145760,watching tv,0,0,a cabinet with a shelf and a mirror on it,No OCR,"house,cup",20190101,1
201901/01/20190101_103821_000,3,2019-01-01,10:38,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.389980,-6.145760,watching tv,0,1,a window with a view of a house in the backgro...,No OCR,"window,house",20190101,1
201901/01/20190101_103853_000,4,2019-01-01,10:38,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.389980,-6.145760,cooking,1,2,a kitchen with a skylight and a bowl of fruit.,No OCR,"human face,girl,dining table,bottle,clothing,p...",20190101,1
201901/01/20190101_103925_000,5,2019-01-01,10:39,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.389980,-6.145760,working on computer,2,3,a computer monitor sitting on top of a desk.,No OCR,"computer keyboard,keyboard,mouse,laptop,tv,book",20190101,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
202006/30/20200630_211803_000,725945,2020-06-30,22:18,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",9737,53.389959,-6.145810,watching tv,100865,283,a person's face in the dark,No OCR,NaN,20200630,1
202006/30/20200630_211908_000,725946,2020-06-30,22:19,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",9737,53.389961,-6.145818,watching tv,100865,283,a picture of a screen with a picture of a pers...,No OCR,"laptop,tv",20200630,1
202006/30/20200630_212119_000,725947,2020-06-30,22:21,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",9737,53.389973,-6.145777,watching tv,100865,283,the dark room is very dark and it is dark and ...,No OCR,NaN,20200630,1


In [22]:
def safe_timezone_lookup(x):
    if pd.notna(x["new_lat"]) and pd.notna(x["new_lng"]):
        try:
            return obj.timezone_at(lat=x["new_lat"], lng=x["new_lng"])
        except Exception as e:
            return None  # or log the error
    else:
        return None

df["timezone"] = df.progress_apply(safe_timezone_lookup, axis=1)

100%|██████████| 725949/725949 [01:31<00:00, 7953.54it/s] 


In [24]:
df.rename(columns={"date": "utc_date", "time": "utc_time"}, inplace=True)

In [32]:
df.set_index("image_id", inplace=True)

In [36]:
def time_to_hhmmss(x):
    if pd.notna(x):
        try:
            time_str = x.split("_")[-2]
            return time_str[:2] + ":" + time_str[2:4] + ":" + time_str[4:6]
        except Exception as e:
            return None  # or log the error
    else:
        return None

df["utc_time"] = df.index.to_series().progress_apply(time_to_hhmmss)

df["utc_date"] = df["utc_date"].astype(str)
df["utc_time"] = df["utc_time"].astype(str)

100%|██████████| 725949/725949 [00:01<00:00, 536863.30it/s]


In [37]:
df.head()

,id,utc_date,utc_time,day_of_week,location,location_displayed,location_id,new_lat,new_lng,activity,activity_id,event_id,caption,ocr,object_tags,video_id,image_available,timezone
image_id,,,,,,,,,,,,,,,,,,
201901/01/20190101_103717_000,1,2019-01-01,10:37:17,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.38998,-6.14576,watching tv,0,0,a window with curtains and a picture of a chair,'口',"curtain,house,bed",20190101,1,Europe/Dublin
201901/01/20190101_103749_000,2,2019-01-01,10:37:49,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.38998,-6.14576,watching tv,0,0,a cabinet with a shelf and a mirror on it,No OCR,"house,cup",20190101,1,Europe/Dublin
201901/01/20190101_103821_000,3,2019-01-01,10:38:21,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.38998,-6.14576,watching tv,0,1,a window with a view of a house in the backgro...,No OCR,"window,house",20190101,1,Europe/Dublin
201901/01/20190101_103853_000,4,2019-01-01,10:38:53,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.38998,-6.14576,cooking,1,2,a kitchen with a skylight and a bowl of fruit.,No OCR,"human face,girl,dining table,bottle,clothing,p...",20190101,1,Europe/Dublin
201901/01/20190101_103925_000,5,2019-01-01,10:39:25,Tuesday,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",0,53.38998,-6.14576,working on computer,2,3,a computer monitor sitting on top of a desk.,No OCR,"computer keyboard,keyboard,mouse,laptop,tv,book",20190101,1,Europe/Dublin


In [38]:
import pandas as pd
import pytz
from datetime import datetime

def convert_to_local_time(row):
    if pd.isna(row["timezone"]):
        return None

    try:
        # Combine UTC day and UTC time into one datetime
        utc_str = f"{row['utc_date']} {row['utc_time']}"
        utc_dt = datetime.strptime(utc_str, "%Y-%m-%d %H:%M:%S")
        utc_dt = pytz.utc.localize(utc_dt)

        # Convert to local time using timezone string
        local_tz = pytz.timezone(row["timezone"])
        local_dt = utc_dt.astimezone(local_tz)

        return local_dt
    except Exception as e:
        return None
    

df["temp_local_time"] = df.progress_apply(convert_to_local_time, axis=1)


100%|██████████| 725949/725949 [00:28<00:00, 25359.95it/s]


In [42]:
df["temp_local_time"].value_counts()
df["local_date"] = df["temp_local_time"].apply(lambda x: str(x).split(" ")[0] if pd.notna(x) else None)
df["local_time"] = df["temp_local_time"].apply(lambda x: str(x).split(" ")[1][:8] if pd.notna(x) else None)

In [48]:
df = df.drop(columns=["temp_local_time"])


In [49]:
cols = [
    "id",
    "utc_date", "utc_time", 
    "timezone",
    "local_date", "local_time",
    "day_of_week",
    "new_lat", "new_lng"
]

# Add any remaining columns not explicitly listed
remaining_cols = [col for col in df.columns if col not in cols]
new_df = df[cols + remaining_cols]

In [55]:
new_df.sample(20)

,id,utc_date,utc_time,timezone,local_date,local_time,day_of_week,new_lat,new_lng,location,location_displayed,location_id,activity,activity_id,event_id,caption,ocr,object_tags,video_id,image_available
image_id,,,,,,,,,,,,,,,,,,,,
202001/14/20200114_181413_000,533908,2020-01-14,18:14:13,Asia/Bangkok,2020-01-15,01:14:13,Tuesday,13.735666,100.556204,"Citadines Sukhumvit 8 Bangkok, Residential Bui...","Citadines Sukhumvit 8 Bangkok, Bangkok, Thailand",7575,working on computer,53287,696,a bed with a computer on top of it.,No OCR,"tv,bed",20200114,1
202005/08/20200508_093903_000,657303,2020-05-08,09:39:03,Europe/Dublin,2020-05-08,10:39:03,Friday,53.385640,-6.257713,"Dublin City University (DCU), College and Univ...","Dublin City University (DCU), Dublin, Ireland,...",9028,working on computer,94838,60,a person is holding a phone in front of a comp...,No OCR,"human face,person,tv",20200508,1
201904/08/20190408_121749_000,128897,2019-04-08,12:17:49,Europe/Dublin,2019-04-08,13:17:49,Monday,53.385524,-6.257882,"Dublin City University (DCU), College and Univ...","Dublin City University (DCU), Dublin, Ireland,...",1895,working on computer,12382,144,a computer monitor sitting on top of a desk.,No OCR,"window,desk,keyboard,cup,laptop,tv,office buil...",20190408,1
201901/02/20190102_141818_000,1690,2019-01-02,14:18:18,Europe/Dublin,2019-01-02,14:18:18,Wednesday,53.153147,-6.915929,"Kildare Village, Outlet Mall, , Kildare, Irela...","Kildare Village, Kildare, Ireland, Kildare — N...",41,eating,138,175,a woman sitting at a table in a restaurant.,No OCR,"human face,dining table,bottle,clothing,person...",20190102,1
201911/19/20191119_111932_000,443073,2019-11-19,11:19:32,Europe/Madrid,2019-11-19,12:19:32,Tuesday,40.547763,-3.692128,"Autonomous University of Madrid, , , Área metr...","Autonomous University of Madrid, Madrid, Commu...",6334,watching tv,40927,195,a person is holding a watch in a room.,No OCR,"human face,person,man",20191119,1
201907/24/20190724_074230_000,255434,2019-07-24,07:42:30,Europe/Dublin,2019-07-24,08:42:30,Wednesday,53.385535,-6.257141,"Dublin City University (DCU), College and Univ...","Dublin City University (DCU), Dublin, Ireland,...",4112,working on computer,24429,47,a computer monitor sitting on top of a desk.,No OCR,"computer monitor,desk,cell phone,cup,tv,fork,o...",20190724,1
201908/29/20190829_220838_000,314748,2019-08-29,22:08:38,Europe/Dublin,2019-08-29,23:08:38,Thursday,53.389948,-6.145831,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",4928,watching tv,30578,349,a picture of a window in a room.,No OCR,"window,picture frame,tv",20190829,1
202004/26/20200426_130339_000,642105,2020-04-26,13:03:39,Europe/Dublin,2020-04-26,14:03:39,Sunday,53.389951,-6.145772,"HOME, , , Dublin, Ireland, Leinster","HOME, Dublin, Ireland, Leinster",8919,Other,93446,53,a laptop computer sitting on top of a desk.,No OCR,NaN,20200426,1
201902/01/20190201_163845_000,43582,2019-02-01,16:38:45,Europe/Dublin,2019-02-01,16:38:45,Friday,53.385505,-6.257230,"Dublin City University (DCU), College and Univ...","Dublin City University (DCU), Dublin, Ireland,...",641,Other,4250,204,a blurry photo of a room with a bunch of cubic...,No OCR,"tv,chair,building",20190201,1


In [57]:
new_df.to_csv("metadata_v4.csv", index=True)

In [58]:
n = pd.read_csv("metadata_v4.csv")
n.head()

,image_id,id,utc_date,utc_time,timezone,local_date,local_time,day_of_week,new_lat,new_lng,...,location_displayed,location_id,activity,activity_id,event_id,caption,ocr,object_tags,video_id,image_available
0,201901/01/20190101_103717_000,1,2019-01-01,10:37:17,Europe/Dublin,2019-01-01,10:37:17,Tuesday,53.38998,-6.14576,...,"HOME, Dublin, Ireland, Leinster",0,watching tv,0,0,a window with curtains and a picture of a chair,'口',"curtain,house,bed",20190101,1
1,201901/01/20190101_103749_000,2,2019-01-01,10:37:49,Europe/Dublin,2019-01-01,10:37:49,Tuesday,53.38998,-6.14576,...,"HOME, Dublin, Ireland, Leinster",0,watching tv,0,0,a cabinet with a shelf and a mirror on it,No OCR,"house,cup",20190101,1
2,201901/01/20190101_103821_000,3,2019-01-01,10:38:21,Europe/Dublin,2019-01-01,10:38:21,Tuesday,53.38998,-6.14576,...,"HOME, Dublin, Ireland, Leinster",0,watching tv,0,1,a window with a view of a house in the backgro...,No OCR,"window,house",20190101,1
3,201901/01/20190101_103853_000,4,2019-01-01,10:38:53,Europe/Dublin,2019-01-01,10:38:53,Tuesday,53.38998,-6.14576,...,"HOME, Dublin, Ireland, Leinster",0,cooking,1,2,a kitchen with a skylight and a bowl of fruit.,No OCR,"human face,girl,dining table,bottle,clothing,p...",20190101,1
4,201901/01/20190101_103925_000,5,2019-01-01,10:39:25,Europe/Dublin,2019-01-01,10:39:25,Tuesday,53.38998,-6.14576,...,"HOME, Dublin, Ireland, Leinster",0,working on computer,2,3,a computer monitor sitting on top of a desk.,No OCR,"computer keyboard,keyboard,mouse,laptop,tv,book",20190101,1
